# Group 12 - API Demo Notebook 

This notebook is an outline of how possible data collection would take place from the [GDELT Doc 2.0 API](https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/). It is by no means perfect, as it only serves as a testing ground to further understand the collection, and processing of data and testing its feasibility. Additionally, it serves as a learning experience and a playground to test new concepts.

**Disclaimer:** Code in this notebook was primarily written by (insert student test number), but A.I. tools (Chatgpt, Copilot) was used to support in exploration, testing and problem-solving.

### GDELT API
We begin by building the code to pull from the GDELT API. To do this we use the gdeltdoc library created by [alex9smith](https://github.com/alex9smith/gdelt-doc-api) on GitHub. The library contains a API wrapper that makes pulling data easy and accessible at scale.

In [1]:
# Importing all relevant libraries
from gdeltdoc import GdeltDoc, Filters,multi_near
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta
import re
import matplotlib.pyplot as plt # Do I still need this for px?
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import tqdm as tq

Creating a dataframe that we will later fill with the data collected

In [2]:
# Dateframe saved under df_art. It will contain a dataframe with information about relevant articles.
df_art = pd.DataFrame(columns=['url',
                               'url_mobile',
                               'title',
                               'seendate',
                               'socialimage',
                               'domain',
                               'language',
                               'sourcecountry'])

Setting our parameters that we can later pass down into the wrapper. We keep this separate so that we can easily make adjustments later.

In [3]:
# Finds words that are in close proximity to one another
near_filter = multi_near([ 
        (5, "muslim", "ban"),
        (5, "travel", "ban"),
        (5, "executive", "order"),
    ])

# The news agencies we are interested in
domains = [
        "nytimes.com",
        "washingtonpost.com",
        "cnn.com",
        "foxnews.com"
    ]

# Keywords that must be in the article
keywords = ["travel ban",
            "muslim ban",
            "executive order 13769", 
            "united states"]

# Amount of dates in our timeframe
dates = range(1,31) # can be expanded to better fit project

We make a function to request data as later we run a loop over each day of the specified timeframe and pull at most 250 articles a day. The API could be set to run for the entirety of the timeframe however, we are limited to 250 articles per request, so looping over each day maximizes our data

In [4]:
# Defining start and end date of our timeframe
start_date = datetime(2017, 1, 1) 
end_date = datetime(2017, 3, 1)

dates = [start_date + timedelta(days=i) for i in range((end_date - start_date).days)]

def fetch_data_for_day(date):
    start = date
    end = start + timedelta(days=1) # pulling for the whole day

    f = Filters(near=near_filter, # Applying our parameters, but we can only have so many or it will fail
                start_date=start.strftime("%Y-%m-%d"),
                end_date=end.strftime("%Y-%m-%d"),
                keyword=keywords,
                domain=domains
                )
    
    gd = GdeltDoc()
    articles = gd.article_search(f) # Pulling the articles
    
    return articles

We utilize ThreadPoolExecutor to make multiple requests at a time which greatly increases the speed at which the loop runs.

In [5]:
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(fetch_data_for_day, date): date for date in dates}
    
    for future in tq.tqdm(as_completed(futures), total=len(futures), desc="Fetching Articles"):
        articles = future.result()
        if not articles.empty:
            df_art = pd.concat([df_art, articles], ignore_index=True)

# We reset the index
df_art.reset_index(drop=True, inplace=True)
print("Done! Total articles collected:", len(df_art))

Fetching Articles: 100%|██████████| 59/59 [00:20<00:00,  2.93it/s]

Done! Total articles collected: 3355


Doing some initial exploratory data analysis.

In [6]:
# Checking the size
df_art.shape

(3355, 8)

In [7]:
df_art.head(10)

,url,url_mobile,title,seendate,socialimage,domain,language,sourcecountry
0,http://www.cnn.com/2017/01/03/politics/dhs-chi...,https://amp.cnn.com/cnn/2017/01/03/politics/dh...,DHS chief : Dont use Dreamer info to deport ...,20170103T214500Z,http://i2.cdn.cnn.com/cnnnext/dam/assets/15020...,cnn.com,English,United States
1,http://www.foxnews.com/politics/2015/10/26/lat...,http://www.foxnews.com/politics/2015/10/26/lat...,Latino nonpartisan super PAC names three Clint...,20170103T084500Z,http://a57.foxnews.com/images.foxnews.com/cont...,foxnews.com,English,United States
2,https://www.washingtonpost.com/news/acts-of-fa...,https://www.washingtonpost.com/amphtml/news/ac...,What could be the biggest religion stories in ...,20170103T143000Z,https://img.washingtonpost.com/rf/image_1484w/...,washingtonpost.com,English,United States
3,http://www.foxnews.com/politics/2015/05/04/ala...,http://www.foxnews.com/politics/2015/05/04/ala...,"Alan Gross would like to go back to Cuba , whe...",20170101T101500Z,http://a57.foxnews.com/images.foxnews.com/cont...,foxnews.com,English,United States
4,http://www.foxnews.com/politics/2015/06/04/gop...,http://www.foxnews.com/politics/2015/06/04/gop...,GOP,20170101T180000Z,http://a57.foxnews.com/images.foxnews.com/cont...,foxnews.com,English,United States
5,http://www.foxnews.com/politics/2015/04/15/str...,http://www.foxnews.com/politics/2015/04/15/str...,Marco Rubio on the Latino vote : Candidates ne...,20170101T050000Z,http://a57.foxnews.com/images.foxnews.com/cont...,foxnews.com,English,United States
6,http://www.foxnews.com/politics/2015/04/08/ven...,http://www.foxnews.com/politics/2015/04/08/ven...,Venezuelans burn Obama effigies as anti,20170101T033000Z,http://a57.foxnews.com/images.foxnews.com/cont...,foxnews.com,English,United States
7,http://www.foxnews.com/politics/2015/04/10/ven...,http://www.foxnews.com/politics/2015/04/10/ven...,Venezuela Maduro claims victory after Obama ba...,20170101T041500Z,http://a57.foxnews.com/images.foxnews.com/cont...,foxnews.com,English,United States
8,http://www.foxnews.com/politics/2015/05/22/us-...,http://www.foxnews.com/politics/2015/05/22/us-...,"U . S ., Cuba say progress made but talks end ...",20170101T151500Z,http://a57.foxnews.com/images.foxnews.com/cont...,foxnews.com,English,United States
9,http://www.foxnews.com/politics/2015/05/19/chr...,http://www.foxnews.com/politics/2015/05/19/chr...,Chris Christie says he opposes path to citizen...,20170101T143000Z,http://a57.foxnews.com/images.foxnews.com/cont...,foxnews.com,English,United States


In [9]:
# Checking what domains were picked up, to make sure that they are within our scope.
print(df_art['domain'].unique())

# Checking what languages the articles were written in, since we only want english.
df_art['language'].value_counts()

['washingtonpost.com' 'cnn.com' 'foxnews.com' 'nytimes.com'
 'nation.foxnews.com' 'money.cnn.com' 'edition.cnn.com'
 'opinionator.blogs.nytimes.com' 'insider.foxnews.com'
 'cnnespanol.cnn.com' 'nytlive.nytimes.com' 'mobile.nytimes.com'
 'apps.washingtonpost.com' 'arabic.cnn.com' 'radio.foxnews.com'
 'us.cnn.com' 'kristof.blogs.nytimes.com' 'live.washingtonpost.com'
 'lens.blogs.nytimes.com']


language
English    3324
Spanish      20
Arabic       11
Name: count, dtype: int64

In [10]:
df_art.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3355 entries, 0 to 3354
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   url            3355 non-null   object
 1   url_mobile     3355 non-null   object
 2   title          3355 non-null   object
 3   seendate       3355 non-null   object
 4   socialimage    3355 non-null   object
 5   domain         3355 non-null   object
 6   language       3355 non-null   object
 7   sourcecountry  3355 non-null   object
dtypes: object(8)
memory usage: 209.8+ KB


Now we take steps to clean the data to further narrow into the scope of the project.

In [11]:
# Printing initial shape of the date to compare
print(df_art.shape)

# Looking for titles that include these words
words_for_art = ["travel", "ban", "immigration", "refugee", 'muslim', 'trump']

# Regex pattern
pattern = r'\b(' + '|'.join(map(re.escape, words_for_art)) + r')\b'

# Filtering
filtered_df = df_art[df_art["title"].str.contains(pattern, case=False, na=False)]

# Articles must be in english and from the U.S.
filtered_df = filtered_df[filtered_df['language'] == 'English']
filtered_df = filtered_df[filtered_df['sourcecountry'] == 'United States']

filtered_df.shape

(3355, 8)


/var/folders/10/vyvj53fd78n53979tcjj1wzc0000gn/T/ipykernel_63752/1942010839.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered_df = df_art[df_art["title"].str.contains(pattern, case=False, na=False)]


(2165, 8)

In [12]:
# Turning 'seendate' into a useable datetime object.
filtered_df["seendate"] = pd.to_datetime(filtered_df["seendate"], format="%Y%m%dT%H%M%SZ", utc=True)

Now we check and delete duplicates. We only delete those that are from the same news agency and have the same title.

In [13]:
# removing duplicates
filtered_df = filtered_df.drop_duplicates(subset=['title', 'domain'], keep = 'first').reset_index(drop=True)

# Checking final shape
filtered_df.shape

(2037, 8)

Now we do some basic initial descriptive statistics through visualizations.

In [14]:
df_art['seendate'] = pd.to_datetime(df_art['seendate'])
daily_counts = df_art.set_index("seendate").resample("d").size() # resample sets it to daily

daily_counts_df = daily_counts.reset_index()
daily_counts_df.columns = ["Date", "Count"]

# Create the line chart
fig = px.line(
    daily_counts_df,
    x="Date",
    y="Count",
    title="Daily Article Frequency",
    template="plotly_white",  # clean background
    markers=True               # adds small dots on each point
)

# Customize
fig.update_traces(line=dict(color="#0074D9", width=2))
fig.update_layout(
    title_font_size=20,
    xaxis_title="Date",
    yaxis_title="Number of Articles",
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor="LightGrey")
)

fig.show()

### Twitter

Because we don't have access to X's Twitter API we simulated some basic data that exponentially increases after a certain date.

In [15]:
# Creates an hourly time range 
dates = pd.date_range("2016-12-20", "2017-02-20 23:00:00", freq="h")

# Setting seed
np.random.seed(1)

base_volume = np.random.normal(loc=100, scale=20, size=len(dates))  # avg 100 tweets/hour, with std 20

# Increase date: "Date of hypothetical event"
increase_start = pd.Timestamp("2017-01-20")
hours_after = (dates - increase_start).total_seconds() / 3600
hours_after = np.maximum(hours_after, 0)

# Expo grow
growth_rate = 0.02  
#exp_increase = 1 + np.exp(growth_rate * hours_after) / 10  # Expo scale
exp_increase = np.where(hours_after > 0, 1 + np.exp(growth_rate * hours_after) / 10, 1)

tweet_volume = base_volume * exp_increase

# Twitter Dateframe
df_tweets = pd.DataFrame({
    "Datetime": dates,
    "Tweet Volume": tweet_volume
})


In [16]:
df_tweets.shape

(1512, 2)

In [17]:
df_tweets.head()

,Datetime,Tweet Volume
0,2016-12-20 00:00:00,132.486907
1,2016-12-20 01:00:00,87.764872
2,2016-12-20 02:00:00,89.436565
3,2016-12-20 03:00:00,78.540628
4,2016-12-20 04:00:00,117.308153


Creating rolling baseline to test.

In [18]:
# Set Datetime as index for rolling calculations
df_tweets.set_index("Datetime", inplace=True)

# Set when the rolling baseline begins. Need to do this otherwise it will trigger to early due to limited info
warmup_days = 20
cutoff = df_tweets.index.min() + pd.Timedelta(days=warmup_days)

# Baseline
rolling_window = 24 * 7  # 7 day in hours
df_tweets["baseline"] = df_tweets["Tweet Volume"].rolling(window=rolling_window, min_periods=1).mean()
df_tweets["rolling_std"] = df_tweets["Tweet Volume"].rolling(window=rolling_window, min_periods=1).std()

# Disable baseline & std for first 20 days so it can get info
df_tweets.loc[df_tweets.index < increase_start, ["baseline", "rolling_std"]] = np.nan

# finding spikes
df_tweets["spike_single"] = ((df_tweets["Tweet Volume"] > df_tweets["baseline"] + 2 * df_tweets["rolling_std"]) & (df_tweets.index >= increase_start))

# Convert to int for rolling sum
df_tweets["spike_int"] = df_tweets["spike_single"].astype(int)

# Finding 2 days of spike
df_tweets["spike"] = df_tweets["spike_int"].rolling(window=72, min_periods=48).sum() == 48

# Clean 
df_tweets.drop(columns="spike_int", inplace=True)



In [19]:
# Making threshold
threshold = 2 # We mention two our paper
df_tweets["spike"] = ((df_tweets["Tweet Volume"] > (df_tweets["baseline"] + threshold * df_tweets["rolling_std"])) & (df_tweets.index >= increase_start))

# finding when it gets triggered
increase_start_time = df_tweets.index[df_tweets["spike"]].min()
print("Tweet volume starts increasing at:", increase_start_time)

str_increase = str(increase_start_time)
match = re.match(r"(\d{4})-(\d{2})-(\d{2})", str_increase)
if match:
    year, month, day = map(int, match.groups())

trend = datetime(year, month, day)
print(year, month, day)

Tweet volume starts increasing at: 2017-01-20 06:00:00
2017 1 20


Graphing tweet data in comparison to article data.

In [ ]:

# Tweets to daily averages
df_daily_tweets = df_tweets["Tweet Volume"].resample("D").mean().reset_index()

fig = go.Figure()

# Articles 
fig.add_trace(
    go.Scatter(
        x=daily_counts_df["Date"],
        y=daily_counts_df["Count"],
        name="Articles",
        line=dict(color="#0074D9", width=3),
        mode="lines+markers",
        yaxis="y1"
    )
)

# Tweets
fig.add_trace(
    go.Scatter(
        x=df_daily_tweets["Datetime"],
        y=df_daily_tweets["Tweet Volume"],
        name="Tweets",
        line=dict(color="#E74C3C", width=2, dash="dot"),
        mode="lines",
        yaxis="y2"
    )
)


fig.update_layout(
    title="Articles vs Tweets (January 2017)",
    template="plotly_white",
    hovermode="x unified",
    xaxis=dict(title="Date",range=[datetime(2017, 1, 1), datetime(2017, 2, 28)]),

    # Articles: left y axis
    yaxis=dict(
        title=dict(text="Number of Articles", font=dict(color="#0074D9")),
        tickfont=dict(color="#0074D9"),
        showgrid=True,
        gridcolor="LightGrey",
        range=[0,300]
    ),

    # Tweets: right y axis
    yaxis2=dict(
        title=dict(text="Tweet Volume", font=dict(color="#E74C3C")),
        tickfont=dict(color="#E74C3C"),
        anchor="x",
        overlaying="y",
        side="right",
        range=[0,500]
    ),

    legend=dict(x=0.02, y=0.98),
)

# event marker for when the Twitter data should start to increase

fig.add_vline(
    x=trend,  
    line_dash="solid",
    line_color="red"
)

fig.add_annotation(
    x=(datetime(2017, 1, 23),'Spike'),
    y=max(daily_counts_df["Count"].max(), df_daily_tweets["Tweet Volume"].max()),  # place at top
    text="Event (Jan 23)",
    showarrow=True,
    arrowhead=2,
    arrowcolor="red",
    yshift=10
)

fig.add_vline(
    x=datetime(2017, 1, 24), # Hardcoding in when a increase happens
    line_dash='dash',
    line_color='blue'
)

fig.show()
